# Willamette: test a un'ora sulle quattro stagioni

**Test completato: il nuovo fit non supera il criterio dichiarato.** La MAE media per
stagione migliora soltanto dello 0,84%; inverno, primavera e autunno peggiorano.
Tutti gli intervalli a blocchi includono zero. Due target estivi sotto i 5 mg/L
illustrativi non vengono anticipati. Nessuna promozione operativa.

Questo notebook riproduce offline il report congelato, senza addestrare un modello,
chiamare USGS o un LLM. Le celle sono state eseguite realmente in ordine con Python
stdlib; non sono output simulati. I comandi separati `verify-fit` e `verify` sono
documentati nel repository e ripetuti dalla CI.


## Context & Methods

Missione: coordinamento ambientale, evidenze verificabili, diritti delle comunità
e dei giovani, finanziamenti legati a risultati verificati. Questa prova riguarda
soltanto ossigeno disciolto a un'ora nella stazione USGS-14211720.

Protocollo pubblicato nel commit eaebc10; modello e implementazione congelati in
b895049 prima del download. Una sola ridge, alpha=100, otto ingressi. Training:
quattro quindicine del 2024. Test: 15–29 gennaio/aprile/luglio/ottobre 2025 UTC.
Il 10% è indicativo. MAE principale e stagioni pesate ugualmente; nessun
peggioramento stagionale osservato e limiti inferiori del bootstrap positivi.

### Key Assumptions

- Fonti archiviate Approved, senza qualifier, con identità/unità fissate; ciò non
  prova disponibilità storica in tempo reale o correttezza di ogni evento fisico.
- Griglia esatta a 30 minuti, 12 misure di storia, target a un'ora; niente
  interpolazione. Copertura minima 90% e almeno 500 coppie sono filtri pratici.
- Bootstrap accoppiato per stagione, blocchi di 48 ore, sensibilità 24/72 ore,
  5.000 repliche con seme fisso. Incertezza esplorativa su finestre brevi.
- Nessuna nuova fascia di previsione calibrata; nessun risultato causale o
  capacità a 24 ore/tre mesi dimostrata.
- Il vecchio fit estivo, appreso/selezionato su agosto 2025, è solo diagnostico:
  usa osservazioni successive alle prime tre finestre del nuovo test.

Riferimenti: [ridge](https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification),
[verifica temporale](https://otexts.com/fpp3/toolbox.html),
[bootstrap temporale](https://otexts.com/fpp3/advanced.html#bootstrapping-time-series).


In [1]:
from pathlib import Path
import gzip, json, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts/experiment_multiseason.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
import experiment_multiseason as trial

folder = ROOT / trial.EVIDENCE
raw = trial.shared.compressed_body(folder / 'evaluation.json.gz')
report = json.loads(raw)
brief = json.loads((folder / 'summary.json').read_bytes())
model = json.loads((folder / 'frozen-model.json').read_bytes())
assert trial.shared.digest(raw) == brief['fullReportRawHash']
assert trial.shared.digest((folder / 'frozen-model.json').read_bytes()) == report['frozenModelHash']
assert model['training']['pairs'] == 2559
print('Training pairs:', model['training']['pairs'])
print('Frozen model hash:', report['frozenModelHash'])
print('Protocol hash:', report['protocolHash'])
print('Full evaluation hash:', brief['fullReportRawHash'])


Training pairs: 2559
Frozen model hash: 0x72d7d14221c7b14c173dba59d69d2a1d36cbb27ee146378b9e046eea998762c9
Protocol hash: 0x40c8b28cc139c8b195a2aacdb5374157b0811186024d2a8370c796f4b9475fb4
Full evaluation hash: 0x0ade7cedf14d71432b828c2e733f395bb4de35c3dda8aa22a39dbe6a03c7567d


## Data

I corpi originali delle risposte USGS sono salvati in gzip con hash dei byte
decompressi. Ogni coppia conserva gli ID delle osservazioni di storia e target.
I manifest legano la cattura al modello già congelato. Gli orari interamente
mancanti rimangono nel denominatore della copertura.


In [2]:
print('Window        pairs / expected  coverage  missing origin / incomplete history-target')
for w in report['windows']:
    grid = w['fullExpectedGrid']
    print(f"{w['window']['id']:13} {len(w['rows']):4} / {grid['expectedOrigins']}       "
          f"{grid['acceptedGridFraction']:.2%}    "
          f"{grid['missingApprovedOxygenOrigins']} / {grid['presentOriginsWithIncompleteHistoryOrTarget']}")
assert sum(len(w['rows']) for w in report['windows']) == 2591
assert all(w['dataScreenPassed'] for w in report['windows'])
print('Total test pairs: 2591 / 2640')


Window        pairs / expected  coverage  missing origin / incomplete history-target
winter-2025    637 / 660       96.52%    12 / 11
spring-2025    647 / 660       98.03%    1 / 12
summer-2025    647 / 660       98.03%    1 / 12
autumn-2025    660 / 660       100.00%    0 / 0
Total test pairs: 2591 / 2640


## Results

MAE = media degli errori assoluti, in mg/L; un vantaggio positivo indica errore
inferiore alla persistenza. La media finale pesa ogni stagione allo stesso modo.
Non si modifica la scelta della metrica dopo aver osservato i risultati.


In [3]:
print('Window         persistence MAE   new MAE     gain percent')
for w in report['windows']:
    base = w['scores']['persistence']['maeMgL']
    new = w['scores']['new']['maeMgL']
    print(f"{w['window']['id']:13} {base:.6f}         {new:.6f}    {100*(1-new/base):+.2f}%")
print('Equal-season means:', report['macro']['seasonalMeanMAEMgL'])
print('Macro gain percent:', report['macro']['relativeGainPercent'])
print('Decision:', report['decision'])
assert report['decision'] == 'observed-seasonal-tradeoff'
assert report['tenPercentIsAcceptanceGate'] is False
assert abs(report['macro']['relativeGainPercent'] - .8378390720197294) < 1e-9


Window         persistence MAE   new MAE     gain percent
winter-2025   0.012873         0.024278    -88.60%
spring-2025   0.027666         0.037157    -34.30%
summer-2025   0.334776         0.308014    +7.99%
autumn-2025   0.068182         0.070332    -3.15%
Equal-season means: {'new': 0.10994522779435503, 'persistence': 0.11087417495288986, 'existingLinear': 0.1377598129808634, 'oldSummer': 0.8045967174013783}
Macro gain percent: 0.8378390720197294
Decision: observed-seasonal-tradeoff


In [4]:
for b in report['bootstrap']:
    print(f"{b['blockHours']}h: {b['nonemptyBlocksPerSeason']} blocks, "
          f"95% interval [{b['lowerGainMgL']:+.6f}, {b['upperGainMgL']:+.6f}] mg/L")
    assert b['lowerGainMgL'] < 0 < b['upperGainMgL']
print('Intervals concern the mean paired error gain, not individual future oxygen values.')


48h: [7, 7, 7, 7] blocks, 95% interval [-0.003821, +0.005414] mg/L
24h: [14, 14, 14, 14] blocks, 95% interval [-0.002845, +0.004437] mg/L
72h: [5, 5, 5, 5] blocks, 95% interval [-0.005161, +0.006190] mg/L
Intervals concern the mean paired error gain, not individual future oxygen values.


### Diagnosi descrittiva dopo il test

I calcoli seguenti non sono nuovi criteri di successo e non addestrano un'altra
variante. Descrivono i valori registrati uguali dopo un'ora e i due target sotto
una soglia illustrativa; non attribuiscono cause fisiche.


In [5]:
for w in report['windows']:
    unchanged = [r for r in w['rows'] if abs(r['actual']-r['persistence']) < 1e-9]
    mae = sum(abs(r['newPrediction']-r['actual']) for r in unchanged) / len(unchanged)
    print(f"{w['window']['id']}: {len(unchanged)}/{len(w['rows'])} unchanged targets, "
          f"new MAE on them {mae:.6f} mg/L; persistence MAE 0")
low = [r for w in report['windows'] for r in w['rows'] if r['actual'] < 5]
assert len(low) == 2
for r in low:
    print({k:r[k] for k in ('origin','target','persistence','actual','newPrediction')})
    print('Target source ID:', r['evidence']['targetOxygen'])
    assert r['newPrediction'] > 5
print('Two archived observations do not validate alert sensitivity or an ecological interpretation.')


winter-2025: 555/637 unchanged targets, new MAE on them 0.015532 mg/L; persistence MAE 0
spring-2025: 482/647 unchanged targets, new MAE on them 0.017100 mg/L; persistence MAE 0
summer-2025: 149/647 unchanged targets, new MAE on them 0.053110 mg/L; persistence MAE 0
autumn-2025: 354/660 unchanged targets, new MAE on them 0.021644 mg/L; persistence MAE 0
{'origin': '2025-07-18T05:00:00.000Z', 'target': '2025-07-18T06:00:00.000Z', 'persistence': 8.5, 'actual': 3.5, 'newPrediction': 8.535660257412193}
Target source ID: c80b70af-704b-4edc-8636-d6cf49410efc
{'origin': '2025-07-20T06:30:00.000Z', 'target': '2025-07-20T07:30:00.000Z', 'persistence': 8.3, 'actual': 3.8, 'newPrediction': 8.293159149289052}
Target source ID: fba3b442-adce-4f3e-86ba-090b56f550f9
Two archived observations do not validate alert sensitivity or an ecological interpretation.


In [6]:
# Full offline reconstruction from source bodies, using frozen weights; no fitting.
maximum_difference = trial.verify_evaluation(ROOT)
assert maximum_difference <= 1e-9
assert report['modelRefittedDuringEvaluation'] is False
assert report['operationalModelChanged'] is False
print('Full report and summary reproduced; maximum numerical difference:', maximum_difference)


Full report and summary reproduced; maximum numerical difference: 0.0


## Takeaways

La diversità stagionale riduce i grossi problemi del vecchio fit estivo, ma il
nuovo modello non batte regolarmente la persistenza e non anticipa i due cali
archiviati. Il test non passa anche senza richiedere il 10%. I periodi stabili e
i bruschi cali sono le difficoltà da capire prima di concordare una nuova prova.

Questi dati ora servono alla diagnosi: non sono più un esame finale mai osservato
per una futura variante. Serviranno un nuovo protocollo e nuovi periodi separati.
Restano invariati LLM, Oracle operativo, TerraLex, quorum, voto, tesoreria e
verifica dei finanziamenti. Non estendiamo automaticamente l'orizzonte.
